Analyse de la structures des bandes électroniques. (TiSnPt)

In [ ]:
from pymatgen import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.symmetry.analyzer import SpacegroupOperations
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.core.operations import SymmOp
from pymatgen.electronic_structure.bandstructure import *
from pymatgen.ext.matproj import MPRester
from jupyter_jsmol.pymatgen import quick_view
from pymatgen.electronic_structure.plotter import*
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with MPRester("kMkuaNd0F7org0EE2QryQaonTKfVvGw4") as m:
    structure = m.get_bandstructure_by_material_id("mp-30847")

electro_band_plot = BSPlotter(structure)

Bande interdite

In [ ]:
band_gap = structure.get_band_gap()
direct = "direct"
if(band_gap['direct'] == False):
    direct = "indirecte"
print("La bande d'énergie est "+direct+", vaut "+str(band_gap['energy'])+" eV la de direction de vecteur k est "+str(band_gap['transition'])+" pour la transition.")

Direction de la dispersion maximale et minimale dans la bande de conduction et de valence

1) Dernière bande de valence et première bande de conduction

In [ ]:
cbmlist = []
vbmlist = []
cbm = structure.get_cbm() 
vbm = structure.get_vbm() 
cbm_band_index = cbm["band_index"].items()
vbm_band_index = vbm["band_index"].items()
for cursor in cbm_band_index:
    cbmlist.append(cursor[1][0])
for cursor in vbm_band_index:
    vbmlist.append(cursor[1][0])

vbm = vbmlist[-1]
cbm = cbmlist[0]
print("dernière bande de valence : "+str(vbm)+".")
print("dernière bande de conduction : "+str(cbm)+".")

2) Directions de dispersion maximale et minimale

In [ ]:
def dEdK(E,k): 
    dEdK = (abs((E[-1]-E[0])))/(k[-1]-k[0])
    result = [E[-1],E[0],k[-1],k[0],dEdK]
    return(np.array(result))

data = electro_band_plot.bs_plot_data()
distance = data.get('ticks').get('distance')
distances = data.get('distances')
energy = data.get('energy')

pente_val = np.zeros((len(distances),5))
pente_cond = np.zeros((len(distances),5))
for i in range(len(distances)): 
    pente_val[i] = dEdK(energy["1"][i][vbm],distances[i])
    pente_cond[i] = dEdK(energy["1"][i][cbm],distances[i])

finder_for_val = np.zeros(len(distances))
finder_for_cond = np.zeros(len(distances))

for j in range(len(distances)):
    finder_for_val[j] = pente_val[j][4]
    finder_for_cond[j] = pente_cond[j][4]

max_val = np.argmax(finder_for_val)
min_val = np.argmin(finder_for_val)
max_cond = np.argmax(finder_for_cond)
min_cond = np.argmin(finder_for_cond)

In [ ]:
electro_band_plot.get_plot()

pente_val_max = pente_val[max_val] 

plt.arrow(pente_val_max[3],pente_val_max[1],pente_val_max[2]-pente_val_max[3],pente_val_max[0]-pente_val_max[1], color='yellow',width = 0.03,length_includes_head =True)

pente_cond_max = pente_cond[max_cond] 

plt.arrow(pente_cond_max[3],pente_cond_max[1],pente_cond_max[2]-pente_cond_max[3],pente_cond_max[0]-pente_cond_max[1], color='orange',width = 0.03,length_includes_head = True)
print("pente maximale (jaune) pour la bande de valence = "+str(pente_val_max[4])+".")
print("pente maximale (orange) pour la bande de conduction = "+str(pente_cond_max[4])+".")

In [ ]:
electro_band_plot.get_plot()

pente_val_min = pente_val[min_val]

plt.arrow(pente_val_min[3],pente_val_min[1],pente_val_min[2]-pente_val_min[3],pente_val_min[0]-pente_val_min[1], color='yellow',width = 0.03,length_includes_head =True)

pente_cond_min = pente_cond[min_cond]

plt.arrow(pente_cond_min[3],pente_cond_min[1],pente_cond_min[2]-pente_cond_min[3],pente_cond_min[0]-pente_cond_min[1], color='orange',width = 0.03,length_includes_head = True)
print(" pente minimale (jaune) pour la bande de valence = "+str(pente_val_min[4])+".")
print("pente minimale (orange) pour la bande de conduction = "+str(pente_cond_min[4])+".")

Masse effective pour la dernière bande de valence et la première bande de conduction

In [ ]:
hbar = (6.62607015*10**(-34))/(2*np.pi) #[J]
eV = 1.602176634*10**(-19) #[J]
hbar_eV = hbar/eV #[eV]

vbms = data.get('vbm')
kval = np.zeros(3)
kval[1] = vbms[0][0]
kval[0] = distances[4][-2]
kval[2] = distances[5][1]

Eval = np.zeros(3)
Eval[1] = vbms[0][1]
Eval[0] = energy.get('1')[4][11][-2]
Eval[2] = energy.get('1')[5][11][1]

interpolation_val = np.polyfit(kval,Eval,2) 
m_eff_val = (vbms[0][0])**2*(hbar_eV**2/(2*interpolation_val[0]))
print("Masse effective pour la bande de valence = "+str(m_eff_val)+" kg.")

cbms = data.get('cbm')

kcond = np.zeros(3)
kcond[1] = cbms[0][0]
kcond[0] = distances[0][-2]
kcond[2] = distances[2][1]

Econd = np.zeros(3)
Econd[1] = vbms[0][1]
Econd[0] = energy.get('1')[0][13][-2]
Econd[2] = energy.get('1')[2][13][1]

interpolation_cond = np.polyfit(kcond,Econd,2) 
m_eff_cond = (cbms[0][0])**2*(hbar_eV**2/(2*interpolation_cond[0])) 
print("Masse effective pour la bande de conduction = "+str(m_eff_cond)+" kg.")